# Exploratory Data Analysis

## Scientific objective
Describe endpoint prevalence, missingness, molecular-weight/logP distributions, scaffold counts, and data limitations without treating visualization as evidence of generalization.

## Inputs
- `data/processed/endpoint_records.csv`

## Expected outputs
- `tables/endpoint_summary.csv`
- `data/processed/descriptors.csv`
- endpoint distribution figures

## Dependencies
RDKit, pandas, matplotlib

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Descriptor distributions are descriptive. They do not establish assay comparability or causal toxicological structure–activity relationships.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Plots can conceal sparse tails and endpoint-specific source effects; numeric summaries are saved alongside figures.

## Next notebook
[07_chemical_space_and_scaffold_analysis.ipynb](./07_chemical_space_and_scaffold_analysis.ipynb)

In [1]:
from pathlib import Path
import os
import json
import warnings
import random

# Must be set before importing NumPy, PyTorch, RDKit, or pyplot.
os.environ["MPLBACKEND"] = "Agg"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import matplotlib
matplotlib.use("Agg", force=True)

# Load pyplot and the Agg renderer before PyTorch and RDKit.
import matplotlib.pyplot as plt

# Force the renderer to initialize now.
_test_figure = plt.figure(figsize=(1, 1))
_test_figure.canvas.draw()
plt.close(_test_figure)

print("Matplotlib backend initialized:", matplotlib.get_backend())

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Run this notebook from the repository root or notebooks directory"
    )

os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])

# This EDA notebook does not require PyTorch seeding.
random.seed(SEED)
np.random.seed(SEED)

print(
    {
        "root": str(ROOT),
        "profile": PROFILE,
        "seed": SEED,
        "matplotlib_backend": matplotlib.get_backend(),
    }
)

Matplotlib backend initialized: Agg
{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723, 'matplotlib_backend': 'Agg'}


In [2]:
from toxicity_screening.descriptors import descriptor_frame

records = pd.read_parquet(
    ROOT / "data" / "processed" / "endpoint_records.parquet"
)

molecules = (
    records[
        ["molecule_id", "standardized_smiles", "scaffold"]
    ]
    .drop_duplicates("molecule_id")
)

desc = descriptor_frame(
    molecules["standardized_smiles"].tolist()
)

desc.insert(
    0,
    "molecule_id",
    molecules["molecule_id"].to_numpy(),
)

desc.to_csv(
    ROOT / "data" / "processed" / "descriptors.csv",
    index=False,
)

summary = (
    records.groupby("endpoint")
    .agg(
        records=("molecule_id", "size"),
        unique_molecules=("molecule_id", "nunique"),
        observed_labels=("label", "count"),
        missing_labels=("label", lambda x: int(x.isna().sum())),
        positives=("label", lambda x: int((x == 1).sum())),
        negatives=("label", lambda x: int((x == 0).sum())),
        positive_prevalence=("label", "mean"),
        unique_scaffolds=("scaffold", "nunique"),
    )
    .reset_index()
)

summary.to_csv(
    ROOT / "tables" / "endpoint_summary.csv",
    index=False,
)

display(summary)

,endpoint,records,unique_molecules,observed_labels,missing_labels,positives,negatives,positive_prevalence,unique_scaffolds
0,SR-ARE,7596,7596,5666,1930,906,4760,0.159901,2269
1,SR-ATAD5,7610,7610,6869,741,260,6609,0.037851,2269
2,SR-MMP,7604,7604,5649,1955,896,4753,0.158612,2268
3,SR-p53,7607,7607,6583,1024,413,6170,0.062737,2268
4,ames_mutagenicity,7246,7246,7246,0,3945,3301,0.544438,1576
5,herg_blockade,12949,12949,12949,0,6476,6473,0.500116,5832


In [3]:
merged = molecules.merge(
    desc,
    on="molecule_id",
    how="inner",
    validate="one_to_one",
)

figure_dir = ROOT / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

print("Molecules before merge:", len(molecules))
print("Descriptor records:", len(desc))
print("Merged records:", len(merged))

for column in ["MolWt", "MolLogP"]:
    if column not in merged.columns:
        print(f"Skipped missing descriptor: {column}")
        continue

    values = (
        pd.to_numeric(merged[column], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .to_numpy(dtype=np.float64)
    )

    print(
        f"{column}: n={len(values):,}, "
        f"min={values.min():.3f}, max={values.max():.3f}"
    )

    output_path = figure_dir / f"eda_{column}.png"

    fig, ax = plt.subplots(figsize=(6, 4))

    ax.hist(values, bins=40)
    ax.set_title(f"{column} distribution")
    ax.set_xlabel(column)
    ax.set_ylabel("Molecules")

    fig.subplots_adjust(
        left=0.13,
        right=0.97,
        bottom=0.15,
        top=0.88,
    )

    fig.savefig(
        output_path,
        dpi=120,
        facecolor="white",
    )

    plt.close(fig)

    print(
        f"Created: {output_path.relative_to(ROOT)} "
        f"({output_path.stat().st_size:,} bytes)"
    )

print("EDA figure generation completed.")

Molecules before merge: 25583
Descriptor records: 25583
Merged records: 25583
MolWt: n=25,583, min=9.012, max=1877.664
Created: figures\eda_MolWt.png (17,295 bytes)
MolLogP: n=25,583, min=-17.406, max=22.612
Created: figures\eda_MolLogP.png (16,328 bytes)
EDA figure generation completed.


### Completion gate
Confirm that the declared artifacts exist before continuing to `07_chemical_space_and_scaffold_analysis.ipynb`.